# 01 Eine erste Frage mit LangChain beantworten

**Leitfrage:** Wie wird aus einer Frage und einem Unterlagentext eine Antwort mit Quelle?

Ein **Sprachmodell** verarbeitet Texte und erzeugt Antworten. Die Abkürzung **LLM** bedeutet *Large Language Model*, also grosses Sprachmodell. Eine Antwort kann richtig klingen und trotzdem falsch sein. Deshalb erhält das Modell hier einen Text als Beleg.

In dieser Reihe entsteht ein kleiner **Modul-Assistent**. Er beantwortet Fragen zu Unterlagen über Bitcoin und Open Banking. **LangChain** ist eine Python-Bibliothek. Sie verbindet Schritte wie das Vorbereiten der Frage und das Lesen der Antwort. LangChain ist selbst kein Sprachmodell. Das Modell wird über die BFH-Schnittstelle aufgerufen.

| Notebook | Neuer Schritt |
|---|---|
| 01 | Eine Frage mit mitgegebenem Text beantworten. |
| 02 | Frühere Nachrichten derselben Unterhaltung mitgeben. |
| 03 | Passende Stellen in zwei PDFs suchen. |
| 04 | Zwischen Suche und Rechenwerkzeug wählen; Ergebnisse prüfen. |

**Frage:** Was soll beantwortet werden? Zum Beispiel: „Auf welche Gesamtmenge ist Bitcoin begrenzt?“

**Prompt:** Der ganze Arbeitsauftrag. Dazu gehören Frage, Regeln und Unterlagentext.

**Modell:** Das Programm, das den Auftrag verarbeitet und die Antwort erzeugt.

Am Ende stehen zwei Felder bereit: `antwort` und `quelle`. Die Quelle muss zum Inhalt passen. Das wird zusätzlich am Text geprüft.

**Literate Programming** bedeutet hier: Erklärungen und Code stehen zusammen. Vor jeder Codezelle wird erklärt, was hineingeht, was geschieht und was herauskommt. Neue Begriffe werden direkt am Beispiel erklärt.


Alle Erklärungen stehen direkt bei den passenden Schritten. Erst den Text lesen, dann die nächste Codezelle ausführen. Die kurzen Verständnisfragen sind zum Nachdenken da; ihre Erklärung lässt sich bei Bedarf aufklappen.


## Lernziele

Nach diesem Notebook können Leserinnen und Leser:

- Regeln, Nutzerfrage und Modellantwort in Nachrichten unterscheiden;
- eine Vorlage mit Platzhaltern verwenden und Vorlage, Modell und Parser verbinden;
- eine fertige Antwort mit einer schrittweise übertragenen Antwort vergleichen;
- gemeldete Tokenzahlen lesen und erklären, warum ohne Tarif kein Geldbetrag berechnet werden kann.


## Konzept Erklärung

Das Modell erhält eine Frage und einen Text. Dieser Text heisst **Kontext**. Eine Regel sagt zum Beispiel: „Antworte nur anhand dieses Textes.“ Regel, Kontext und Frage bilden den Prompt.

LangChain verpackt den Prompt in **Nachrichten**. Jede Nachricht hat eine Rolle:

| Rolle | Bedeutung hier |
|---|---|
| System | Regeln für die Antwort. |
| Human | Text und Frage der nutzenden Person. |
| AI | Zurückgegebene Modellantwort. |

**Text und Frage → Nachrichten → BFH-Modell → Antwortnachricht**

Eine neue Anfrage enthält frühere Antworten nicht automatisch. Wenn sie gebraucht werden, müssen sie erneut mitgeschickt werden. Notebook 02 ergänzt diesen Verlauf.

### Der Weg zur Antwort als Bild

![Regeln, Text und Frage werden durch die Vorlage zu Nachrichten. Das BFH-Modell erzeugt eine Antwort. Der Parser liest diese ein und prüft die Felder antwort und quelle.](abbildungen/01_von_der_frage_zur_antwort.svg)

Das Bild zeigt den gewünschten Ablauf der Kette am Ende dieses Notebooks. Die **Vorlage** setzt die Eingaben zusammen. Das **Modell** formuliert die Antwort mit den verlangten Feldern. Der **Parser** liest und prüft diese Felder; er entscheidet nicht, ob die Aussage fachlich richtig ist. Zuerst werden diese Schritte einzeln angesehen.


## Setup

Ein **Kernel** ist der laufende Python-Prozess im Notebook. Er merkt sich Werte zwischen den Zellen. Die Zellen deshalb von oben nach unten ausführen. Nach einem Neustart oben beginnen.

Zuerst werden Hilfsmittel geladen. `os` liest Einstellungen, `sys` liefert Informationen zu Python. `Path` baut Dateipfade. `version` liest Paketversionen. `load_dotenv` lädt später die lokale Konfiguration. Diese Imports senden keine Anfrage.


In [19]:
import os
import sys
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

Die folgende Prüfung erlaubt Python 3.12 und 3.13. `assert` bedeutet: Die Bedingung muss stimmen. Sonst erscheint eine Fehlermeldung.

Danach werden Python- und LangChain-Version angezeigt. So lässt sich die verwendete Umgebung später nachvollziehen. Ein Modell wird noch nicht aufgerufen.


In [20]:
assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 oder 3.13 erforderlich'
print('Python:', sys.version.split()[0])
print('LangChain Core:', version('langchain-core'))

Python: 3.13.15
LangChain Core: 1.7.3


Die Dateien liegen im Projektordner. `Path.cwd()` liest den Ordner, in dem Jupyter gestartet wurde. Ist das der Unterordner `notebooks`, führt `.parent` eine Ebene nach oben.

Python soll auch das lokale Paket `modul_assistent` finden. Deshalb wird der Projektordner in `sys.path` aufgenommen. Das ist die Liste der Ordner, in denen Python nach Imports sucht. Das Ergebnis ist ein verwendbarer Projektpfad.


In [21]:
projektordner = Path.cwd()
if projektordner.name == 'notebooks':
    projektordner = projektordner.parent
assert (projektordner / 'modul_assistent' / 'core.py').is_file(), 'Im Projekt oder in notebooks starten'
if str(projektordner) not in sys.path:
    sys.path.insert(0, str(projektordner))


Die Datei `.env` enthält lokale BFH-Einstellungen. `load_dotenv` übernimmt sie in die Umgebung des Python-Prozesses. `override=False` bedeutet: Bereits gesetzte Werte bleiben erhalten.

Eingabe ist der Dateipfad. Ergebnis sind verfügbare Einstellungen. Es wird noch keine Anfrage gesendet. Der Schlüssel wird nicht angezeigt.


In [22]:
load_dotenv(projektordner / '.env', override=False)


True

## Experiment

### 1 Die Bitcoin Unterlage finden

Die erste Frage verwendet `Eigenschaften-Bitcoin.pdf`. Sie liegt unter `data/modulunterlagen`. Die Datei hat eine Seite.

Der Pfad wird aus dem Projektordner aufgebaut. `is_file()` prüft, ob dort eine Datei liegt. Angezeigt wird nur ihr Name. Die Open-Banking-PDF kommt später bei der Suche hinzu.


In [23]:
pdf_datei = projektordner / 'data' / 'modulunterlagen' / 'Eigenschaften-Bitcoin.pdf'
assert pdf_datei.is_file(), 'Die Bitcoin-Unterlage fehlt in data/modulunterlagen.'
print('Unterlage:', pdf_datei.name)

Unterlage: Eigenschaften-Bitcoin.pdf


### 2 Den Text der ersten Seite auslesen

`PdfReader` öffnet die gefundene PDF. `pages[0]` wählt die erste Seite: Python zählt Positionen ab null. `extract_text()` liest den Text aus.

Der Text wird unter dem Namen `kontext` gespeichert. Ein solcher benannter Wert heisst **Variable**. Die Prüfung erkennt leeren Text. Danach wird der tatsächlich gelesene Text angezeigt.

Hier wird die ganze Seite verwendet. Die Suche nach einer passenden Stelle folgt erst in Notebook 03.


In [24]:
from pypdf import PdfReader

pdf = PdfReader(pdf_datei)
kontext = pdf.pages[0].extract_text()
assert kontext and kontext.strip(), 'Aus der PDF wurde kein Text gelesen.'
print(kontext)

Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Millionen Bitcoins 
begrenzt. 
• Anteile an Bitcoins oder ganze Bitcoins sind immer klar einer Ad- 
resse zugeordnet.  
Informationen 
zu Transaktio- 
nen von Bit- 
coins 
• Transaktionen finden im Bitcoin-System direkt von Person zu Per- 
son statt. 
• Der zwischengeschalteter Zahlungsdienstleister (Bank)

### 3 Die Frage und die Quelle festlegen

Die Frage nennt die gesuchte Bitcoin-Gesamtmenge. Die Quelle nennt Datei und Seite. `f'...'` setzt einen vorhandenen Wert in einen Text ein. Hier wird so der Dateiname ergänzt.

Frage, Quelle und Kontext stehen in getrennten Variablen. Dadurch lässt sich später zum Beispiel die Frage ändern, ohne den PDF-Leseschritt zu ändern.


In [25]:
frage = 'Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?'
quelle = f'{pdf_datei.name}, S. 1'
print('Frage:', frage)
print('Quelle:', quelle)

Frage: Auf welche Gesamtmenge ist Bitcoin laut der Unterlage begrenzt?
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 4 Die Modellschnittstelle vorbereiten

`chat_model` ist eine Funktion im Projekt. Sie erstellt den Baustein für BFH-Anfragen. Dieser Baustein heisst hier **Modellobjekt**.

Dafür verwendet die Funktion den **Adapter** `ChatOpenAI`. Ein Adapter übersetzt zwischen zwei Schnittstellen. Hier übersetzt er LangChain-Nachrichten in Anfragen an die BFH-API. Eine **API** ist die Schnittstelle, über die Programme miteinander sprechen.

Zuerst wird die Funktion nur importiert. Es wird keine Anfrage gesendet.

Dokumentation: [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai), [BFH Inference](https://infra.pages.ti.bfh.ch/mlmp/src/llm/).


In [26]:
from modul_assistent.core import chat_model


Die Verbindung braucht Modellname, API-Adresse und Schlüssel. `os.getenv` liest jeweils eine Einstellung. Der zweite Wert ist der Ersatzwert, falls die Einstellung fehlt. `.strip()` entfernt Leerzeichen am Anfang und Ende.

Für den Schlüssel gibt es keinen nutzbaren Ersatzwert. Die nächste Zelle liest die Angaben nur lokal. Sie sendet und zeigt den Schlüssel nicht an.


In [27]:
bfh_modell = os.getenv('BFH_CHAT_MODEL', 'qwen3.8-27b').strip()
bfh_adresse = os.getenv('BFH_BASE_URL', 'https://inference.proxy.ti.bfh.ch/v1').strip()
bfh_schluessel = os.getenv('BFH_API_KEY', '').strip()


### BFH Adresse und Modell kontrollieren

Die API-Adresse nennt den Dienst und seinen Pfad. Der Modellname nennt ein Modell dieses Dienstes. Beides muss zusammenpassen. Die beiden Werte werden jetzt angezeigt.

Der hier verwendete BFH-Pfad endet auf `/v1`. Ein falscher Pfad kann HTTP 404 auslösen. Das bedeutet: Die angefragte Ressource wurde nicht gefunden. Auch die Meldung `OpenAIModelNotFoundError` beweist allein keinen falschen Modellnamen. Zuerst Adresse samt Pfad und Modellnamen prüfen.


In [28]:
print('BFH-API-Adresse:', bfh_adresse)
print('BFH-Modell:', bfh_modell)

BFH-API-Adresse: https://inference.proxy.ti.bfh.ch/v1
BFH-Modell: qwen3.8-27b


Vor dem Aufruf werden leere Angaben erkannt. `if` führt einen Schritt nur aus, wenn seine Bedingung stimmt. `raise ValueError` bricht mit einer verständlichen Meldung ab.

`urlparse` zerlegt die Adresse. `scheme` ist etwa `https`; `netloc` enthält den Servernamen. Beide müssen vorhanden sein. Diese Prüfung arbeitet lokal. Sie beweist noch nicht, dass der Dienst den Schlüssel akzeptiert oder das Modell anbietet.


In [29]:
from urllib.parse import urlparse

if not bfh_schluessel:
    raise ValueError('BFH_API_KEY fehlt in der lokalen Konfiguration.')
if not bfh_modell:
    raise ValueError('BFH_CHAT_MODEL darf nicht leer sein.')
adressteile = urlparse(bfh_adresse)
if adressteile.scheme not in ('http', 'https') or not adressteile.netloc:
    raise ValueError('BFH_BASE_URL benötigt eine vollständige HTTP- oder HTTPS-Adresse.')


`chat_model()` baut nun das BFH-Modellobjekt auf. Die Klammern rufen die Funktion auf. Sie übernimmt die Einstellungen aus `core.py`.

Dort sind auch Wartezeit und begrenzte Wiederholungsversuche festgelegt. Eine geringe Zufälligkeit erleichtert Vergleiche. Sie garantiert keinen identischen Wortlaut.

Ergebnis ist `modell`, ein Baustein für spätere Anfragen. Erst `modell.invoke(...)` oder `modell.stream(...)` sendet Nachrichten.


In [30]:
modell = chat_model()


### 5 Eine erste Anfrage senden

Nun werden zwei Nachrichten vorbereitet. `SystemMessage` enthält die Regel. `HumanMessage` enthält Text und Frage. Eckige Klammern bilden eine **Liste**: mehrere Werte in einer festen Reihenfolge.

`modell.invoke(nachrichten)` sendet diese Liste an BFH und wartet auf das Ergebnis. **invoke** bedeutet hier: den Baustein mit dieser Eingabe ausführen. Das Ergebnis ist eine `AIMessage`. Ihr Feld `content` enthält den Antworttext.

**Nachrichtenliste → BFH-Aufruf → Antworttyp und Antworttext**


**Kurz überlegen:** Was wird an das BFH-Modell geschickt?

<details>
<summary>Erklärung ansehen</summary>

Die Nachrichten mit Regeln, Unterlagentext und Frage. Das Modell bekommt nicht automatisch den ganzen Projektordner.

</details>


In [31]:
from langchain_core.messages import SystemMessage, HumanMessage

nachrichten = [
    SystemMessage(content='Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.'),
    HumanMessage(content=f'Text: {kontext}\nFrage: {frage}'),
]

erste_antwort = modell.invoke(nachrichten)
print('Antworttyp:', type(erste_antwort).__name__)
print('Antwort:', erste_antwort.content)

Antworttyp: AIMessage
Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf **21 Millionen Bitcoins** begrenzt.


### 6 Was enthält eine Anschlussfrage?

Die neue Frage lautet: „Und wie entsteht der Wechselkurs?“ Ohne Verlauf steht nur dieser Satz in der Anfrage. Mit Verlauf stehen auch die Regeln, der Bitcoin-Text und die echte erste Antwort darin.

`+` verbindet hier zwei Listen. Die Schleifen zeigen jede enthaltene Nachricht einzeln. `type` benennt ihre Rolle, `content` ihren Text. `len` zählt die Nachrichten. Diese Zelle sendet keine neue Anfrage.


In [32]:
anschlussfrage = HumanMessage(content='Und wie entsteht der Wechselkurs?')
ohne_verlauf = [anschlussfrage]
mit_verlauf = nachrichten + [erste_antwort, anschlussfrage]
print('Ohne Verlauf:')
for nachricht in ohne_verlauf:
    print(nachricht.type, nachricht.content)
print('Mit Verlauf:')
for nachricht in mit_verlauf:
    print(nachricht.type, nachricht.content)
assert len(ohne_verlauf) == 1
assert len(mit_verlauf) == 4


Ohne Verlauf:
human Und wie entsteht der Wechselkurs?
Mit Verlauf:
system Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.
human Text: Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Millionen Bitcoins 
begrenzt. 
• Anteile an Bitcoins oder ganze Bitcoins sind immer klar einer Ad- 
resse zugeordnet.  
Informatio

### 7 Die Antwort schrittweise anzeigen

`invoke()` wartet auf die fertige Antwort. **Streaming** liefert schon Teile, während die Antwort entsteht. `stream()` gibt diese Teile nacheinander zurück.

Die `for`-Schleife zeigt jeden Teil sofort an. `end=''` verhindert einen neuen Absatz nach jedem Teil. `flush=True` veranlasst die sofortige Ausgabe.

Das ist eine weitere BFH-Anfrage. Sie zeigt nicht bloss die gespeicherte erste Antwort erneut an. Wortlaut und Laufzeit können sich ändern.


In [33]:
for teil in modell.stream(nachrichten):
    print(teil.content, end='', flush=True)
print()



Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf **21 Millionen Bitcoins** begrenzt.


### 8 Tokenverbrauch und Kosten einordnen

Ein **Token** ist ein Textbaustein des Modells. Er ist nicht immer ein ganzes Wort. Auch Regeln und Kontext gehören zur Eingabe.

`usage_metadata` enthält Nutzungsdaten, wenn der Dienst sie liefert. Die Schlüssel `input_tokens` und `output_tokens` benennen Eingabe und Ausgabe. Eckige Klammern schlagen den Wert zu einem Schlüssel nach.

Die Ausgabe betrifft nur die erste vollständige Anfrage. Fehlen die Daten, wird das offen angezeigt. Dann ist kein tatsächlicher Verbrauch bekannt.


In [34]:
nutzung = erste_antwort.usage_metadata

if nutzung:
    print('Eingabe-Tokens:', nutzung['input_tokens'])
    print('Ausgabe-Tokens:', nutzung['output_tokens'])
else:
    print('Keine Tokenzahlen vom Modell gemeldet.')

Eingabe-Tokens: 579
Ausgabe-Tokens: 120


Die BFH-API meldet die verwendeten Eingabe- und Ausgabe-Tokens. Für dieses Projekt liegt kein Preis pro Token vor. Deshalb lässt sich daraus kein tatsächlicher Geldbetrag berechnen.

Allgemein hängen Kosten von der Tokenzahl und dem jeweiligen Tarif ab. Ohne bekannten Tarif bleibt diese Berechnung offen. Das bedeutet nicht automatisch, dass der Dienst kostenlos ist.


### 9 Die gewünschte Antwortform festlegen

Bisher ist die Antwort freier Text. Jetzt soll sie zwei benannte Felder haben: `antwort` und `quelle`. Ein **Schema** beschreibt diese gewünschte Form.

Die Klasse `Antwort` legt die Felder fest. `str` bedeutet: Der Wert muss Text sein. Pydantic ist die Bibliothek, die diese Prüfung übernimmt.

Ein **OutputParser** liest den Modelltext und prüft ihn gegen das Schema. Fehlt ein Feld, kann er einen Fehler melden. Eine bestandene Prüfung beweist noch keinen richtigen Bitcoin-Inhalt.


**Kurz überlegen:** Die Antwort hat beide Felder. Ist sie damit auch inhaltlich richtig?

<details>
<summary>Erklärung ansehen</summary>

Nein. Die Felder zeigen nur die Form. Die Aussage muss zusätzlich mit der angegebenen Quelle verglichen werden.

</details>


In [35]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

class Antwort(BaseModel):
    antwort: str
    quelle: str

parser = PydanticOutputParser(pydantic_object=Antwort)
print('Erwartete Felder:', list(Antwort.model_fields))

Erwartete Felder: ['antwort', 'quelle']


### 10 Eine wiederverwendbare Promptvorlage erstellen

Eine **Vorlage** ist ein Text mit freien Stellen. `{frage}` wird zum Beispiel durch die aktuelle Frage ersetzt. So kann derselbe Auftrag mit anderen Eingaben verwendet werden.

`ChatPromptTemplate.from_messages` erstellt eine Vorlage für System- und Human-Nachricht. Die kurze deutsche Formatvorgabe verlangt genau die beiden Textfelder `antwort` und `quelle`. `.partial(...)` setzt diesen festen Teil schon jetzt ein. Andere Platzhalter bleiben noch offen.

`eingaben` ist ein **Dictionary**: eine Zuordnung von Schlüsseln zu Werten. Unter dem Schlüssel `'frage'` steht die Variable `frage`. Die Schlüssel müssen zu den Platzhaltern passen.

Dokumentation: [ChatPromptTemplate](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate).


In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Antworte auf Deutsch und nur anhand des Textes. '
     'Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. '
     'Ansonsten übernimm die mitgegebene Quellenangabe. {formatvorgabe}'),
    ('human', 'Text: {kontext}\nQuelle: {quelle}\nFrage: {frage}'),
]).partial(formatvorgabe='Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.')

eingaben = {'kontext': kontext, 'quelle': quelle, 'frage': frage}

`prompt.invoke(eingaben)` führt hier nur die Vorlage aus. Es ersetzt ihre Platzhalter durch die Dictionary-Werte. Ein Modell wird dabei nicht aufgerufen.

`to_string()` macht die entstandenen Nachrichten lesbar. So lässt sich vor dem Senden prüfen: Stehen Frage, Text, Quelle und Formatregel an der richtigen Stelle?


In [37]:
vorbereiteter_prompt = prompt.invoke(eingaben)
print(vorbereiteter_prompt.to_string())

System: Antworte auf Deutsch und nur anhand des Textes. Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. Ansonsten übernimm die mitgegebene Quellenangabe. Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.
Human: Text: Eigenschaften Bitcoin 
Meisser, Luzius, Hauser-Spühler, Gabriel 
https://doi.org/10.5167/uzh-153361 
Allgemeine In- 
formationen zu 
Bitcoin 
• Bitcoin gehört zu den 20 grössten Währungen 
=>gemessen an Geldmenge 
• Die Kryptowährung «Bitcoin» ist auf der «Blockchain» aufgebaut. 
• Bitcoins fallen ebenfalls in die Konkursmasse bei einer Liquidie- 
rung eines Unternehmens oder einer Privatperson. 
Was ist Bitcoin? • Bitcoin ist eine dezentral organisierte Internetwährung 
• Der Wechselkurs entsteht wie bei anderen Währungen durch An- 
gebot und Nachfrage auf dem freien Markt. 
• Der Wert eines Bitcoins wird von niemandem garantiert. 
• Die Menge an Bitcoins ist vordefiniert und auf 21 Mill

### 11 Prompt Modell und Parser verbinden

Eine **Chain** ist eine Kette von Arbeitsschritten. Hier sind es drei:

**Eingabe-Dictionary → Promptvorlage → Modell → Parser → Ergebnis mit zwei Feldern**

LangChain nennt diese Schreibweise **LCEL**: LangChain Expression Language. Der Strich `|` gibt das Ergebnis links an den nächsten Baustein rechts weiter.

Das Modell soll **JSON** ausgeben. JSON ist ein Textformat für benannte Daten, etwa `{"antwort": "Text", "quelle": "Datei"}`. Der Parser liest diesen Text und prüft die Felder. `chain.invoke(eingaben)` führt die ganze Kette aus und sendet dabei eine neue BFH-Anfrage.


`bind` ergänzt eine feste Einstellung für die folgenden Aufrufe. Hier bittet sie die BFH-Schnittstelle um ein JSON-Objekt. Der Parser prüft danach die beiden Felder.


**Kurz überlegen:** Welcher Schritt erzeugt die Antwort?

<details>
<summary>Erklärung ansehen</summary>

Das Modell. Die Vorlage stellt die Nachrichten zusammen. Der Parser liest danach die Antwort und prüft die Felder.

</details>


In [38]:
struktur_modell = modell.bind(response_format={'type': 'json_object'})
chain = prompt | struktur_modell | parser
ergebnis = chain.invoke(eingaben)

print('Antwort:', ergebnis.antwort)
print('Quelle:', ergebnis.quelle)
assert isinstance(ergebnis, Antwort)


Antwort: Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.
Quelle: Eigenschaften-Bitcoin.pdf, S. 1


### 12 Den Datenfluss Schritt für Schritt untersuchen

Dieselben Schritte lassen sich einzeln ausführen. Zuerst wird die Vorlage befüllt. Dann wird eine weitere BFH-Anfrage gesendet. Zuletzt liest der Parser die Antwort.

Die Ausgaben zeigen Zwischenstände. `type(...).__name__` zeigt den Namen eines Datentyps. `model_dump()` zeigt die geprüften Felder als Dictionary.

Für den geforderten Debugger-Durchlauf einen **Haltepunkt** bei `nachrichtenwert` setzen. Ein Haltepunkt hält die Ausführung an dieser Zeile an. In Jupyter mit aktiviertem Debugger lässt sich danach zeilenweise weitergehen und der Variableninhalt ansehen. So wird sichtbar, in welchem Schritt ein Fehler entsteht. Die neue Modellantwort kann anders lauten als zuvor.


In [39]:
nachrichtenwert = prompt.invoke(eingaben)
modellantwort = struktur_modell.invoke(nachrichtenwert)
gepruefte_antwort = parser.invoke(modellantwort)

print('Nach der Vorlage:', type(nachrichtenwert).__name__)
print('Nach dem Modell:', modellantwort.content)
print('Nach dem Parser:', gepruefte_antwort.model_dump())

Nach der Vorlage: ChatPromptValue
Nach dem Modell: {
  "antwort": "Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.",
  "quelle": "Eigenschaften-Bitcoin.pdf, S. 1"
}
Nach dem Parser: {'antwort': 'Laut der Unterlage ist die Menge an Bitcoins vordefiniert und auf 21 Millionen Bitcoins begrenzt.', 'quelle': 'Eigenschaften-Bitcoin.pdf, S. 1'}


## Häufige Fehler

### Eine benötigte Promptvariable fehlt

Die Vorlage braucht `kontext`, `quelle` und `frage`. Jetzt wird absichtlich nur `frage` übergeben. Das ist eine unvollständige Eingabe.

Erwartet wird ein **KeyError**: Ein benötigter Schlüssel fehlt. Der Fehler entsteht beim Befüllen der Vorlage. Es wird noch keine BFH-Anfrage gesendet.

`try` versucht den falschen Schritt. `except KeyError` fängt genau diesen erwarteten Fehler ab und zeigt ihn. `else` würde einen unerwartet fehlerfreien Durchlauf melden. Der geplante Fehler hält das Notebook deshalb nicht auf.


In [40]:
try:
    prompt.invoke({'frage': frage})
except KeyError as fehler:
    print('Erwarteter Fehler:', type(fehler).__name__)
    print(str(fehler)[:450])
else:
    raise AssertionError('Der erwartete Fehler ist nicht aufgetreten.')

Erwarteter Fehler: KeyError
"Input to ChatPromptTemplate is missing variables {'kontext', 'quelle'}.  Expected: ['frage', 'kontext', 'quelle'] Received: ['frage']\nNote: if you intended {kontext} to be part of the string and not a variable, please escape it with double curly braces like: '{{kontext}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


### Die Eingaben vervollständigen

Die Korrektur verwendet das vollständige Dictionary `eingaben`. Es enthält alle drei benötigten Schlüssel.

Die Vorlage wird erneut befüllt. Die Prüfung erwartet zwei Nachrichten: System und Human. Eine zusätzliche Modellanfrage ist dafür nicht nötig.


In [41]:
korrigierter_prompt = prompt.invoke(eingaben)
print('Die Vorlage wurde vollständig ausgefüllt.')
assert len(korrigierter_prompt.to_messages()) == 2

Die Vorlage wurde vollständig ausgefüllt.


## Prompt Logbuch

### Warum steht dieser Abschnitt hier?

Die Aufgabenstellung verlangt, die Entwicklung eines Prompts zu dokumentieren: ein zuerst selbst geschriebener Auftrag, die später mit KI überarbeitete Fassung und eine kurze Erklärung der Änderung. So wird sichtbar, was einen genaueren Prompt ausmacht.

Ein **Prompt** besteht aus Anweisungen, Frage und mitgegebenen Informationen. Hier werden die erste dokumentierte Fassung und die aktuelle Endfassung gegenübergestellt. „Endfassung“ bedeutet die derzeit verwendete Version, nicht eine perfekte Anweisung.

**Herkunft:** Die hier vorbereiteten Texte entstanden mit KI-Unterstützung. Ein unabhängig davon selbst geschriebener Erstprompt ist nicht belegt. Es werden keine eigenen Entwürfe oder Versuchsergebnisse nachträglich erfunden.

### 1 Erste dokumentierte Fassung

Die erste Fassung steht bei der ersten Modellanfrage. Die Endfassung steht in der späteren Promptvorlage. Beide sind im Notebook als Code vorhanden und wurden technisch über BFH ausgeführt; sie sind keine nachgewiesene selbstständige Erstfassung der Autorin.

```text
System:
Antworte auf Deutsch und nur anhand des Textes. Fehlt die Antwort, sage dies ausdrücklich.

Human:
Text: {kontext}
Frage: {frage}
```

### 2 Aktuelle Endfassung

```text
System:
Antworte auf Deutsch und nur anhand des Textes. Fehlt die Information, sage dies ausdrücklich und verwende als Quelle keine Quelle. Ansonsten übernimm die mitgegebene Quellenangabe. Gib nur ein JSON-Objekt mit den Textfeldern antwort und quelle zurück. Kein Schema und keine Liste.

Human:
Text: {kontext}
Quelle: {quelle}
Frage: {frage}
```

`{kontext}` ist der gelesene PDF-Text, `{quelle}` enthält Datei und Seite, `{frage}` die aktuelle Frage. Die Beispiel-Frage betrifft die Begrenzung von Bitcoin.

### 3 Was wurde genauer?

Die erste Fassung begrenzt die Antwort auf den Text. Die Endfassung legt zusätzlich Quelle, Verhalten bei fehlender Information und die zwei JSON-Felder fest. So weiss das Modell genauer, was erwartet wird. Der Parser prüft die Felder; die fachliche Aussage muss trotzdem am Text geprüft werden.

Die Beschreibung erklärt die Änderungen an der Anweisung. Sie behauptet keine gemessene Verbesserung zwischen zwei Fassungen, wenn ein solcher Vergleich nicht ausgeführt wurde.


## Zusammenfassung

### Was wurde aufgebaut?

Eine erste Frage zur Bitcoin-Unterlage wurde über die BFH-API an ein Sprachmodell geschickt. Dafür erhielt das Modell Regeln, den Text aus der PDF und die Frage. Diese Informationen bilden den **Prompt**, also den Arbeitsauftrag.

Die Nachrichten haben verschiedene Rollen: **System** enthält die Regeln, **Human** die Nutzerfrage mit dem Text und **AI** die Antwort des Modells.

Danach wurden drei Schritte zu einer **Chain**, also einer Verarbeitungskette, verbunden:

**Vorlage füllen → Modell antwortet → Parser liest die Antwort**

Die Vorlage setzt Frage und Text an die vorgesehenen Stellen. Das Modell formuliert die Antwort. Der Parser prüft, ob die Felder `antwort` und `quelle` vorhanden sind. Ob die Aussage durch die Quelle belegt ist, muss zusätzlich gelesen werden.

### Was sollte daran verständlich werden?

Eine Modellanfrage ist ein nachvollziehbarer Weg von Eingaben zu einem Ergebnis. Die Antwort kann fertig oder per **Streaming** in Teilen angezeigt werden. Die gemeldeten **Tokens** beschreiben die verarbeitete Textmenge. Ohne bekannten Tarif lässt sich daraus kein tatsächlicher Geldbetrag berechnen.

Der absichtlich unvollständige Aufruf zeigte ausserdem: Fehlt eine benötigte Angabe, kann die Vorlage nicht richtig ausgefüllt werden.

### Was kommt als Nächstes?

Eine neue Anfrage enthält frühere Nachrichten nicht automatisch. Notebook 02 ergänzt deshalb einen gespeicherten Gesprächsverlauf.

### Lernziele überprüfen

- Welche Regeln, Texte und Fragen bekommt das Modell?
- Was macht die Vorlage, was das Modell und was der Parser?
- Warum kann eine formal richtige Antwort trotzdem falsch sein?
- Was zeigen Streaming und Tokenzahlen über den Aufruf?


## Übungsaufgabe

Formuliere zwei neue Fragen zur Bitcoin-Seite. Eine soll der Text beantworten können. Für die andere soll eine Angabe fehlen. Ein Beispiel für fehlende Information ist der heutige Bitcoin-Kurs.

Schreibe für einen neuen Fall zuerst selbst einen Prompt. Lass ihn erst danach mit KI-Hilfe überarbeiten. Führe die tatsächlichen Fassungen aus und ergänze das Prompt-Logbuch.

Prüfe am PDF-Text: Stimmt die Antwort? Stimmt die Quelle? Wird fehlende Information benannt? Die letzte Codezelle bleibt für die eigene Lösung leer.
